In [2]:
import csv
import hashlib
import json
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False
    print("[WARN] ollama package not installed. Install with: pip install ollama")


In [3]:
DATASET_PATHS = {
    "DoS":   Path("../../Dataset/DoS_dataset_clean.csv"),
    "Fuzzy": Path("../../Dataset/Fuzzy_dataset_clean.csv"),
    "Gear":  Path("../../Dataset/gear_dataset_clean.csv"),
    "RPM":   Path("../../Dataset/RPM_dataset_clean.csv"),
}

WINDOW_SIZE = 200
WINDOW_STRIDE = 400  # increase stride to skip more windows
SAMPLE_RATIO = 0.10  # sample fewer windows
SHORT_ANSWERS_PER_WINDOW = 1  # minimal per-window questions
GLOBAL_SEED = 50

BYTE_COLUMNS = [f"Byte{i}" for i in range(1, 9)]
ATTACK_LABELS = ["DoS", "Fuzzy", "Gear", "RPM"]
SUPPRESSION_THRESHOLD = 1e-3
FLOODING_THRESHOLD = 5e-4
FLOODING_STD = 5e-4
JITTER_RATIO = 0.5
EXPECTED_COLUMNS = ["Timestamp", "ID", "DLC", *BYTE_COLUMNS, "Flag"]

SELECTED_DATASETS = ["DoS", "Fuzzy", "Gear", "RPM"]  # enable all types

CACHE_DIR = Path("cache")

# SA template thresholds (global, applied across datasets)
ATTACK_BURST_MIN_LEN = 5
ATTACK_SUSTAIN_LEN = 20
MAX_ID_SHARE_LOW = 0.2
MAX_ID_SHARE = 0.5
FPS_SUBWINDOW = 50
PAYLOAD_VAR_LOW_PCTL = 33
PAYLOAD_VAR_HIGH_PCTL = 66
HIGH_DLC_LOW_PCTL = 50
HIGH_DLC_MID_PCTL = 80
HIGH_DLC_HIGH_PCTL = 95
OUT_OF_RANGE_ID_RATIO_PCTL = 90
CRITICAL_SPIKE_MAX_COUNT = 10
BASELINE_P95_PERCENTILE = 95
BASELINE_P99_PERCENTILE = 99
BASELINE_P1_PERCENTILE = 1
BASELINE_P99_PERCENTILE = 99
TIMESTAMP_RESOLUTION = 1e-4
TIMESTAMP_BUCKET_SHARE_THRESHOLD = 0.30

# %%
# LLM Configuration for Answer Enhancement
LLM_ENABLED = False  # disable for speed; set True when you want LLM answers
LLM_MODEL = "mistral"  # Change to your preferred model (e.g., "llama2", "neural-chat", "mistral")
LLM_BASE_URL = "http://localhost:11434"  # Default Ollama URL
MAX_ANSWER_TOKENS = 150  # Limit answer length
LLM_TEMPERATURE = 0.7  # Creativity level (0=deterministic, 1=creative)

if LLM_ENABLED:
    print("[INFO] LLM enhancement ENABLED")
    print(f"[INFO] Using model: {LLM_MODEL}")
    print(f"[INFO] Ollama URL: {LLM_BASE_URL}")
    print("[INFO] Make sure Ollama is running: ollama serve")
else:
    print("[WARN] LLM enhancement DISABLED - ollama package not available or turned off")


[WARN] LLM enhancement DISABLED - ollama package not available or turned off


In [4]:
def _normalize_flag_series(series: pd.Series) -> pd.Series:
    mapped = series.map({"R": 0, "T": 1})
    numeric = pd.to_numeric(series, errors="coerce")
    combined = mapped.fillna(numeric).fillna(0).astype(int)
    return combined


def _to_int_hex(val) -> int:
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s.lower().startswith("0x"):
        s = s[2:]
    try:
        return int(s, 16)
    except Exception:
        try:
            return int(float(val))
        except Exception:
            return 0


def load_datasets(paths: Dict[str, Path]):
    datasets: Dict[str, pd.DataFrame] = {}
    profiles: Dict[str, dict] = {}

    for name, path in paths.items():
        csv_path = Path(path)
        if not csv_path.exists():
            print(f"[WARN] Dataset {name} not found at {csv_path}, skipping.")
            continue

        df = pd.read_csv(csv_path)
        if not set(EXPECTED_COLUMNS).issubset(df.columns):
            # Raw CarHackingDataset files ship without headers; add them when missing
            df = pd.read_csv(csv_path, header=None, names=EXPECTED_COLUMNS)
        else:
            df = df.rename(columns=str)

        missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
        for col in missing_cols:
            df[col] = 0
        df = df[EXPECTED_COLUMNS]

        df["Timestamp"] = pd.to_numeric(df["Timestamp"], errors="coerce")
        df["ID"] = df["ID"].apply(_to_int_hex)
        df["DLC"] = df["DLC"].apply(_to_int_hex)
        for col in BYTE_COLUMNS:
            df[col] = df[col].apply(_to_int_hex)

        df["Flag"] = _normalize_flag_series(df["Flag"])
        df = df.dropna(subset=["Timestamp", "ID"]).reset_index(drop=True)

        datasets[name] = df
        profiles[name] = {
            "expected_ids": set(),
            "critical_ids": set(),
            "attack_label": name,
        }
        print(f"[INFO] Loaded {name}: {len(df)} rows.")

    return datasets, profiles


@dataclass
class BaselineStats:
    expected_ids: Set[int]
    critical_ids: Set[int]
    frame_rate_p95: float
    frame_rate_p99: float
    payload_var_low: float
    payload_var_high: float
    high_dlc_low: float
    high_dlc_mid: float
    high_dlc_high: float
    critical_rate_p95_by_id: Dict[int, float]
    byte_range_by_id: Dict[int, tuple]
    out_of_range_id_ratio_high: float


def _safe_percentile(values: List[float], percentile: float, default: float) -> float:
    if not values:
        return default
    return float(np.percentile(values, percentile))


def iter_window_starts(num_rows: int) -> List[int]:
    if num_rows < WINDOW_SIZE:
        return []
    return list(range(0, num_rows - WINDOW_SIZE + 1, WINDOW_STRIDE))


def _dataset_signature(paths: Dict[str, Path]) -> str:
    parts = [
        f"WINDOW_SIZE={WINDOW_SIZE}",
        f"WINDOW_STRIDE={WINDOW_STRIDE}",
        f"BASELINE_P1_PERCENTILE={BASELINE_P1_PERCENTILE}",
        f"BASELINE_P99_PERCENTILE={BASELINE_P99_PERCENTILE}",
        f"BASELINE_P95_PERCENTILE={BASELINE_P95_PERCENTILE}",
        f"BASELINE_P99_PERCENTILE={BASELINE_P99_PERCENTILE}",
        f"PAYLOAD_VAR_LOW_PCTL={PAYLOAD_VAR_LOW_PCTL}",
        f"PAYLOAD_VAR_HIGH_PCTL={PAYLOAD_VAR_HIGH_PCTL}",
        f"HIGH_DLC_LOW_PCTL={HIGH_DLC_LOW_PCTL}",
        f"HIGH_DLC_MID_PCTL={HIGH_DLC_MID_PCTL}",
        f"HIGH_DLC_HIGH_PCTL={HIGH_DLC_HIGH_PCTL}",
        f"OUT_OF_RANGE_ID_RATIO_PCTL={OUT_OF_RANGE_ID_RATIO_PCTL}",
    ]
    for name, path in sorted(paths.items()):
        csv_path = Path(path)
        if csv_path.exists():
            stat = csv_path.stat()
            parts.append(f"{name}:{csv_path.resolve()}:{stat.st_size}:{stat.st_mtime}")
        else:
            parts.append(f"{name}:{csv_path.resolve()}:missing")
    return "|".join(parts)


def _run_signature(paths: Dict[str, Path]) -> str:
    parts = [
        _dataset_signature(paths),
        f"SAMPLE_RATIO={SAMPLE_RATIO}",
        f"GLOBAL_SEED={GLOBAL_SEED}",
        f"SHORT_ANSWERS_PER_WINDOW={SHORT_ANSWERS_PER_WINDOW}",
        f"ATTACK_BURST_MIN_LEN={ATTACK_BURST_MIN_LEN}",
        f"ATTACK_SUSTAIN_LEN={ATTACK_SUSTAIN_LEN}",
        f"MAX_ID_SHARE_LOW={MAX_ID_SHARE_LOW}",
        f"MAX_ID_SHARE={MAX_ID_SHARE}",
        f"FPS_SUBWINDOW={FPS_SUBWINDOW}",
        f"SUPPRESSION_THRESHOLD={SUPPRESSION_THRESHOLD}",
        f"FLOODING_THRESHOLD={FLOODING_THRESHOLD}",
        f"FLOODING_STD={FLOODING_STD}",
        f"JITTER_RATIO={JITTER_RATIO}",
        f"CRITICAL_SPIKE_MAX_COUNT={CRITICAL_SPIKE_MAX_COUNT}",
        f"TIMESTAMP_RESOLUTION={TIMESTAMP_RESOLUTION}",
        f"TIMESTAMP_BUCKET_SHARE_THRESHOLD={TIMESTAMP_BUCKET_SHARE_THRESHOLD}",
    ]
    return "|".join(parts)


def load_cached_baseline(cache_path: Path) -> Optional[BaselineStats]:
    if not cache_path.exists():
        return None
    try:
        with cache_path.open("rb") as f:
            baseline = pickle.load(f)
    except Exception:
        return None
    required = [
        "expected_ids",
        "critical_ids",
        "frame_rate_p95",
        "frame_rate_p99",
        "payload_var_low",
        "payload_var_high",
        "high_dlc_low",
        "high_dlc_mid",
        "high_dlc_high",
        "critical_rate_p95_by_id",
        "byte_range_by_id",
        "out_of_range_id_ratio_high",
    ]
    if not all(hasattr(baseline, name) for name in required):
        return None
    return baseline


def save_cached_baseline(cache_path: Path, baseline: BaselineStats) -> None:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as f:
        pickle.dump(baseline, f)


def load_cache(cache_path: Path, default):
    if not cache_path.exists():
        return default
    with cache_path.open("rb") as f:
        return pickle.load(f)


def save_cache(cache_path: Path, payload) -> None:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("wb") as f:
        pickle.dump(payload, f)


def compute_global_baseline(datasets: Dict[str, pd.DataFrame]) -> BaselineStats:
    all_frames = pd.concat(list(datasets.values()), ignore_index=True)
    id_counts = all_frames["ID"].value_counts()
    expected_ids = set(int(x) for x in id_counts.head(5).index.tolist())
    critical_ids = set(int(x) for x in id_counts.head(3).index.tolist())

    byte_range_by_id: Dict[int, tuple] = {}
    for can_id, group in all_frames.groupby("ID"):
        values = group[BYTE_COLUMNS].to_numpy().flatten()
        if values.size == 0:
            low = high = 0.0
        else:
            low = float(np.percentile(values, BASELINE_P1_PERCENTILE))
            high = float(np.percentile(values, BASELINE_P99_PERCENTILE))
        byte_range_by_id[int(can_id)] = (low, high)

    frame_rate_samples: List[float] = []
    dominant_var_samples: List[float] = []
    high_dlc_share_samples: List[float] = []
    out_of_range_id_ratio_samples: List[float] = []
    critical_rate_samples: Dict[int, List[float]] = {cid: [] for cid in critical_ids}

    for name, df in datasets.items():
        starts = iter_window_starts(len(df))
        for start in tqdm(starts, desc=f"Baseline {name} windows"):
            window = df.iloc[start:start + WINDOW_SIZE]
            if len(window) < WINDOW_SIZE:
                continue
            duration = float(window["Timestamp"].iloc[-1] - window["Timestamp"].iloc[0])
            if duration <= 0:
                continue
            frame_rate_samples.append(len(window) / duration)

            id_counts_window = window["ID"].value_counts()
            dominant_id = int(id_counts_window.index[0])
            dom_group = window[window["ID"] == dominant_id]
            dom_payloads = dom_group[BYTE_COLUMNS].to_numpy().flatten()
            if dom_payloads.size > 0:
                dominant_var_samples.append(float(np.var(dom_payloads)))

            high_dlc_share_samples.append(float((window["DLC"] >= 8).mean()))

            unique_ids = set(int(x) for x in window["ID"].unique())
            out_of_range_ids = 0
            for cid in unique_ids:
                cid_group = window[window["ID"] == cid]
                payloads = cid_group[BYTE_COLUMNS].to_numpy()
                low, high = byte_range_by_id.get(cid, (0.0, 0.0))
                if payloads.size > 0 and ((payloads < low) | (payloads > high)).any():
                    out_of_range_ids += 1
            if unique_ids:
                out_of_range_id_ratio_samples.append(out_of_range_ids / len(unique_ids))

            for cid in critical_ids:
                count = int((window["ID"] == cid).sum())
                critical_rate_samples[cid].append(count / duration)

    frame_rate_p95 = _safe_percentile(frame_rate_samples, BASELINE_P95_PERCENTILE, 0.0)
    frame_rate_p99 = _safe_percentile(frame_rate_samples, BASELINE_P99_PERCENTILE, 0.0)
    payload_var_low = _safe_percentile(dominant_var_samples, PAYLOAD_VAR_LOW_PCTL, 0.0)
    payload_var_high = _safe_percentile(dominant_var_samples, PAYLOAD_VAR_HIGH_PCTL, payload_var_low)
    high_dlc_low = _safe_percentile(high_dlc_share_samples, HIGH_DLC_LOW_PCTL, 0.0)
    high_dlc_mid = _safe_percentile(high_dlc_share_samples, HIGH_DLC_MID_PCTL, high_dlc_low)
    high_dlc_high = _safe_percentile(high_dlc_share_samples, HIGH_DLC_HIGH_PCTL, high_dlc_mid)
    out_of_range_id_ratio_high = _safe_percentile(out_of_range_id_ratio_samples, OUT_OF_RANGE_ID_RATIO_PCTL, 0.0)

    critical_rate_p95_by_id = {
        cid: _safe_percentile(samples, BASELINE_P95_PERCENTILE, 0.0)
        for cid, samples in critical_rate_samples.items()
    }

    return BaselineStats(
        expected_ids=expected_ids,
        critical_ids=critical_ids,
        frame_rate_p95=frame_rate_p95,
        frame_rate_p99=frame_rate_p99,
        payload_var_low=payload_var_low,
        payload_var_high=payload_var_high,
        high_dlc_low=high_dlc_low,
        high_dlc_mid=high_dlc_mid,
        high_dlc_high=high_dlc_high,
        critical_rate_p95_by_id=critical_rate_p95_by_id,
        byte_range_by_id=byte_range_by_id,
        out_of_range_id_ratio_high=out_of_range_id_ratio_high,
    )


rng_global = np.random.default_rng(GLOBAL_SEED)
dataset_paths = {k: v for k, v in DATASET_PATHS.items() if k in SELECTED_DATASETS}
datasets, profiles = load_datasets(dataset_paths)

baseline_signature = _dataset_signature(dataset_paths)
baseline_hash = hashlib.sha256(baseline_signature.encode("utf-8")).hexdigest()[:12]
baseline_cache_path = CACHE_DIR / f"baseline_{baseline_hash}.pkl"
baseline = load_cached_baseline(baseline_cache_path)
if baseline is None:
    baseline = compute_global_baseline(datasets)
    save_cached_baseline(baseline_cache_path, baseline)
    print(f"[INFO] Saved baseline cache -> {baseline_cache_path}")
else:
    print(f"[INFO] Loaded baseline cache -> {baseline_cache_path}")

run_signature = _run_signature(dataset_paths)
run_hash = hashlib.sha256(run_signature.encode("utf-8")).hexdigest()[:12]

for name in profiles:
    profiles[name]["expected_ids"] = baseline.expected_ids
    profiles[name]["critical_ids"] = baseline.critical_ids
print(f"[INFO] Global baseline: {len(baseline.expected_ids)} expected IDs, "
      f"{len(baseline.critical_ids)} critical IDs.")


[INFO] Loaded DoS: 3665771 rows.
[INFO] Loaded Fuzzy: 3838860 rows.
[INFO] Loaded Gear: 4443142 rows.
[INFO] Loaded RPM: 4621702 rows.


Baseline RPM windows: 100%|██████████| 11554/11554 [03:14<00:00, 59.53it/s]


[INFO] Saved baseline cache -> cache\baseline_d92527824db7.pkl
[INFO] Global baseline: 5 expected IDs, 3 critical IDs.


In [5]:
# Cell 3: helpers (window, stats)
def sample_window_indices(starts: List[int], rng: np.random.Generator) -> List[int]:
    if not starts:
        return []
    sample_size = max(1, int(len(starts) * SAMPLE_RATIO))
    sample_size = min(sample_size, len(starts))
    return sorted(rng.choice(starts, size=sample_size, replace=False))


def format_window(df: pd.DataFrame) -> str:
    rows = []
    for _, row in df.iterrows():
        byte_vals = [int(row[col]) for col in BYTE_COLUMNS]
        rows.append(
            f"Timestamp={row['Timestamp']:.6f} | "
            f"ID={int(row['ID'])} | DLC={int(row['DLC'])} | "
            f"bytes={byte_vals} | Flag={int(row['Flag'])} |"
        )
    return "\n".join(rows)


def compute_basic_stats(df: pd.DataFrame) -> dict:
    stats = {}
    total_frames = len(df)
    if total_frames == 0:
        return stats

    id_counts = df["ID"].value_counts()
    stats["id_counts"] = id_counts
    max_count = int(id_counts.max()) if not id_counts.empty else 0
    dominant_ids = id_counts[id_counts == max_count].index.tolist() if max_count > 0 else []
    stats["dominant_id"] = int(min(dominant_ids)) if dominant_ids else 0
    stats["dominant_share"] = (max_count / total_frames) if total_frames > 0 else 0.0

    # timing
    if total_frames > 1:
        diffs = df["Timestamp"].to_numpy()[1:] - df["Timestamp"].to_numpy()[:-1]
        stats["diffs"] = diffs
        stats["gap_max"] = float(diffs.max())
        stats["gap_min"] = float(diffs.min())
        stats["gap_mean"] = float(diffs.mean())
        stats["gap_std"] = float(diffs.std())
        stats["window_duration"] = float(df["Timestamp"].iloc[-1] - df["Timestamp"].iloc[0])
    else:
        stats["diffs"] = np.array([])
        stats["gap_max"] = 0.0
        stats["gap_min"] = 0.0
        stats["gap_mean"] = 0.0
        stats["gap_std"] = 0.0
        stats["window_duration"] = 0.0

    # payload const for dominant id
    dom_group = df[df["ID"] == stats["dominant_id"]]
    if not dom_group.empty:
        payload = dom_group[BYTE_COLUMNS].to_numpy()
        stats["dominant_payload_var"] = float(payload.var())
    else:
        stats["dominant_payload_var"] = 0.0

    return stats


def ratio_to_percent_str(ratio: float) -> str:
    pct = ratio * 100
    if abs(pct - round(pct)) < 1e-9:
        return str(int(round(pct)))
    return f"{pct:.2f}".rstrip("0").rstrip(".")


def ratio_to_str(value: float) -> str:
    return f"{value:.4f}".rstrip("0").rstrip(".")


def _calc_indicator_counts(df: pd.DataFrame, baseline: BaselineStats) -> dict:
    total = len(df)
    if total == 0:
        return {"indicator_count": 0, "normal_count": 0}
    id_counts = df["ID"].value_counts()
    max_share = float(id_counts.iloc[0] / total)
    attack_count = int((df["Flag"] != 0).sum())
    timestamps = df["Timestamp"].to_numpy()
    diffs = np.diff(timestamps) if total > 1 else np.array([])
    gap_mean = float(diffs.mean()) if diffs.size > 0 else 0.0
    gap_std = float(diffs.std()) if diffs.size > 0 else 0.0
    max_gap = float(diffs.max()) if diffs.size > 0 else 0.0
    backward_steps = bool((diffs < 0).any()) if diffs.size > 0 else False
    if TIMESTAMP_RESOLUTION > 0:
        rounded = np.round(timestamps / TIMESTAMP_RESOLUTION) * TIMESTAMP_RESOLUTION
        bucket_counts = pd.Series(rounded).value_counts()
        bucket_share = float(bucket_counts.iloc[0] / total) if not bucket_counts.empty else 0.0
    else:
        bucket_share = 0.0

    high_dlc_share = float((df["DLC"] >= 8).mean()) if total > 0 else 0.0
    present_ids = set(int(x) for x in df["ID"].unique())
    out_of_range_id_count = 0
    for cid in present_ids:
        group = df[df["ID"] == cid]
        payloads = group[BYTE_COLUMNS].to_numpy()
        if payloads.size == 0:
            continue
        low, high = baseline.byte_range_by_id.get(cid, (0.0, 0.0))
        if ((payloads < low) | (payloads > high)).any():
            out_of_range_id_count += 1

    timing_anomaly = (
        (gap_mean < FLOODING_THRESHOLD)
        or (max_gap > SUPPRESSION_THRESHOLD)
        or (gap_mean > 0 and gap_std > JITTER_RATIO * gap_mean)
    )
    timestamp_anomaly = backward_steps or (bucket_share >= TIMESTAMP_BUCKET_SHARE_THRESHOLD)

    indicator_flags = [
        attack_count > 0,
        max_share > MAX_ID_SHARE,
        high_dlc_share > baseline.high_dlc_high,
        out_of_range_id_count > 0,
        timing_anomaly,
        timestamp_anomaly,
    ]
    indicator_count = int(sum(1 for flag in indicator_flags if flag))

    normal_flags = [
        attack_count == 0,
        max_share <= MAX_ID_SHARE_LOW,
        high_dlc_share <= baseline.high_dlc_low,
        out_of_range_id_count == 0,
        (gap_mean >= FLOODING_THRESHOLD)
        and (max_gap <= SUPPRESSION_THRESHOLD)
        and (gap_mean == 0 or gap_std <= JITTER_RATIO * gap_mean),
        not timestamp_anomaly,
    ]
    normal_count = int(sum(1 for flag in normal_flags if flag))

    return {"indicator_count": indicator_count, "normal_count": normal_count}


def compute_window_metrics(df: pd.DataFrame, baseline: BaselineStats) -> dict:
    total = len(df)
    if total == 0:
        return {}

    flags = df["Flag"].to_numpy()
    attack_count = int((flags != 0).sum())
    attack_ratio = attack_count / total
    max_run = 0
    run = 0
    for flag in flags:
        if flag != 0:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0

    id_counts = df["ID"].value_counts()
    max_count = int(id_counts.max()) if not id_counts.empty else 0
    dominant_ids = id_counts[id_counts == max_count].index.tolist() if max_count > 0 else []
    dominant_id = int(min(dominant_ids)) if dominant_ids else 0
    max_share = float(max_count / total) if total > 0 else 0.0

    timestamps = df["Timestamp"].to_numpy()
    duration = float(timestamps[-1] - timestamps[0]) if total > 0 else 0.0
    fps = float(total / duration) if duration > 0 else 0.0

    mid = total // 2
    fps_first_half = 0.0
    fps_second_half = 0.0
    if mid > 1:
        t0 = float(timestamps[0])
        t_mid = float(timestamps[mid - 1])
        d1 = t_mid - t0
        if d1 > 0:
            fps_first_half = mid / d1
    if total - mid > 1:
        t_mid2 = float(timestamps[mid])
        t_end = float(timestamps[-1])
        d2 = t_end - t_mid2
        if d2 > 0:
            fps_second_half = (total - mid) / d2

    max_subwindow_fps = fps
    if FPS_SUBWINDOW > 1 and total >= FPS_SUBWINDOW:
        max_subwindow_fps = 0.0
        for start in range(0, total - FPS_SUBWINDOW + 1):
            sub = df.iloc[start:start + FPS_SUBWINDOW]
            t0 = float(sub["Timestamp"].iloc[0])
            t1 = float(sub["Timestamp"].iloc[-1])
            d = t1 - t0
            if d > 0:
                rate = len(sub) / d
                if rate > max_subwindow_fps:
                    max_subwindow_fps = rate

    diffs = np.diff(timestamps) if total > 1 else np.array([])
    gap_mean = float(diffs.mean()) if diffs.size > 0 else 0.0
    gap_std = float(diffs.std()) if diffs.size > 0 else 0.0
    max_gap = float(diffs.max()) if diffs.size > 0 else 0.0
    backward_steps = bool((diffs < 0).any()) if diffs.size > 0 else False

    if TIMESTAMP_RESOLUTION > 0:
        rounded = np.round(timestamps / TIMESTAMP_RESOLUTION) * TIMESTAMP_RESOLUTION
        bucket_counts = pd.Series(rounded).value_counts()
        bucket_share = float(bucket_counts.iloc[0] / total) if not bucket_counts.empty else 0.0
    else:
        bucket_share = 0.0

    dom_group = df[df["ID"] == dominant_id]
    dom_payloads = dom_group[BYTE_COLUMNS].to_numpy().flatten()
    dominant_var = float(np.var(dom_payloads)) if dom_payloads.size > 0 else 0.0

    high_dlc_share = float((df["DLC"] >= 8).mean()) if total > 0 else 0.0
    dlc_mode = int(df["DLC"].mode().iloc[0]) if total > 0 else 0

    present_ids = set(int(x) for x in df["ID"].unique())
    missing_count = sum(1 for eid in baseline.expected_ids if eid not in present_ids)

    critical_rates: Dict[int, float] = {}
    critical_counts: Dict[int, int] = {}
    if duration > 0:
        for cid in baseline.critical_ids:
            count = int((df["ID"] == cid).sum())
            critical_counts[cid] = count
            critical_rates[cid] = count / duration

    critical_rate_threshold = max(baseline.critical_rate_p95_by_id.values() or [0.0])
    max_critical_rate = max(critical_rates.values()) if critical_rates else 0.0
    critical_above = [
        cid for cid, rate in critical_rates.items()
        if rate > critical_rate_threshold
    ]

    out_of_range_id_count = 0
    for cid in present_ids:
        group = df[df["ID"] == cid]
        payloads = group[BYTE_COLUMNS].to_numpy()
        if payloads.size == 0:
            continue
        low, high = baseline.byte_range_by_id.get(cid, (0.0, 0.0))
        if ((payloads < low) | (payloads > high)).any():
            out_of_range_id_count += 1
    out_of_range_id_ratio = out_of_range_id_count / len(present_ids) if present_ids else 0.0

    frame_cols = ["Timestamp", "ID", "DLC", "Flag"] + BYTE_COLUMNS
    frame_counts = df[frame_cols].value_counts()
    duplicate_count = int((frame_counts[frame_counts > 1] - 1).sum()) if not frame_counts.empty else 0

    indicator_counts = _calc_indicator_counts(df, baseline)
    first_half = df.iloc[:mid] if mid > 0 else df
    second_half = df.iloc[mid:] if total - mid > 0 else df
    ind_first = _calc_indicator_counts(first_half, baseline)["indicator_count"]
    ind_second = _calc_indicator_counts(second_half, baseline)["indicator_count"]
    contradictions = indicator_counts["indicator_count"] > 0 and indicator_counts["normal_count"] > 0

    return {
        "attack_count": attack_count,
        "attack_ratio": attack_ratio,
        "attack_max_run": max_run,
        "dominant_id": dominant_id,
        "max_share": max_share,
        "fps": fps,
        "fps_first_half": fps_first_half,
        "fps_second_half": fps_second_half,
        "max_subwindow_fps": max_subwindow_fps,
        "gap_mean": gap_mean,
        "gap_std": gap_std,
        "max_gap": max_gap,
        "backward_steps": backward_steps,
        "bucket_share": bucket_share,
        "dominant_var": dominant_var,
        "high_dlc_share": high_dlc_share,
        "dlc_mode": dlc_mode,
        "missing_count": missing_count,
        "max_critical_rate": max_critical_rate,
        "critical_above": critical_above,
        "critical_counts": critical_counts,
        "out_of_range_id_count": out_of_range_id_count,
        "out_of_range_id_ratio": out_of_range_id_ratio,
        "duplicate_count": duplicate_count,
        "indicator_count": indicator_counts["indicator_count"],
        "indicator_count_first_half": ind_first,
        "indicator_count_second_half": ind_second,
        "contradictions": contradictions,
    }


# Cell 4: Short Answer question generation
def generate_sa_dominant_id(df: pd.DataFrame, stats: dict) -> Optional[dict]:
    """
    Which CAN ID appears most frequently in this window?
    """
    dom_id = stats.get("dominant_id")
    if dom_id is None:
        return None
    
    id_counts = stats.get("id_counts")
    if id_counts is None or id_counts.empty:
        return None
    
    count = int(id_counts.iloc[0])
    total = int(id_counts.sum())
    
    return {
        "type": "dominant_id",
        "question": "Which CAN ID appears most frequently in this window?",
        "answer": f"0x{dom_id:03X}",
        "explanation": f"ID 0x{dom_id:03X} appears {count} times out of {total} total frames."
    }


def generate_sa_attack_type(stats: dict, profile: dict) -> Optional[dict]:
    """
    What attack type best fits this window?
    """
    gt_label = profile.get("attack_label", "DoS")
    return {
        "type": "attack_type",
        "question": "What attack type best fits the behavior in this window?",
        "answer": gt_label,
        "explanation": f"This window is labeled as {gt_label} attack."
    }


def generate_sa_timing_pattern(stats: dict) -> Optional[dict]:
    """
    Describe the timing behavior in this window.
    """
    gap_mean = stats.get("gap_mean", 0.0)
    gap_std = stats.get("gap_std", 0.0)
    gap_max = stats.get("gap_max", 0.0)
    
    if gap_mean < FLOODING_THRESHOLD and gap_std < FLOODING_THRESHOLD:
        pattern = "Flooding - extremely small gaps with very consistent timing"
    elif gap_max > SUPPRESSION_THRESHOLD * 5:
        pattern = "Suppression - several large gaps indicating dropped or delayed frames"
    elif gap_std < gap_mean * 0.1:
        pattern = "Uniform - timing is mostly consistent with minimal jitter"
    else:
        pattern = "Irregular - timing varies unpredictably"
    
    return {
        "type": "timing_pattern",
        "question": "Describe the timing behavior observed in this window.",
        "answer": pattern,
        "explanation": f"Mean gap: {gap_mean:.6f}s, Std: {gap_std:.6f}s, Max gap: {gap_max:.6f}s"
    }


def generate_sa_missing_id(df: pd.DataFrame, profile: dict) -> Optional[dict]:
    """
    Is any expected control ID missing from this window?
    """
    expected_ids: Set[int] = profile.get("expected_ids", set())
    if not expected_ids:
        return None
    
    present_ids = set(int(x) for x in df["ID"].unique())
    missing = [eid for eid in expected_ids if eid not in present_ids]
    
    if missing:
        missing_str = ", ".join([f"0x{eid:03X}" for eid in missing])
        answer = f"Yes, {missing_str}"
    else:
        answer = "No, all expected control IDs are present"
    
    return {
        "type": "expected_id_missing",
        "question": "Is any expected control ID missing from this window?",
        "answer": answer,
        "explanation": f"Expected IDs: {', '.join([f'0x{x:03X}' for x in expected_ids])}, Present: {', '.join([f'0x{x:03X}' for x in present_ids])}"
    }


def generate_sa_dlc_summary(df: pd.DataFrame) -> Optional[dict]:
    """
    Summarize the DLC (Data Length Code) distribution in this window.
    """
    if "DLC" not in df.columns:
        return None
    
    dlc = df["DLC"].to_numpy()
    if dlc.size == 0:
        return None
    
    mean_dlc = float(dlc.mean())
    unique_dlcs = sorted(np.unique(dlc))
    most_common_dlc = int(np.argmax(np.bincount(dlc.astype(int))))
    
    summary = f"DLC values range from {int(dlc.min())} to {int(dlc.max())}, with mean {mean_dlc:.2f} and most common value {most_common_dlc}"
    
    return {
        "type": "dlc_summary",
        "question": "Summarize the DLC (Data Length Code) distribution in this window.",
        "answer": summary,
        "explanation": f"DLC statistics: min={int(dlc.min())}, max={int(dlc.max())}, mean={mean_dlc:.2f}, mode={most_common_dlc}"
    }


def generate_sa_payload_variance(df: pd.DataFrame, stats: dict) -> Optional[dict]:
    """
    What does the payload variance suggest about this window?
    """
    dom_id = stats.get("dominant_id")
    if dom_id is None:
        return None
    
    var = stats.get("dominant_payload_var", 0.0)
    
    if var < 1e-3:
        interpretation = "The dominant ID transmits identical payloads repeatedly, suggesting constant sensor readings or a crafted/synthetic attack."
    elif var < 1.0:
        interpretation = "The dominant ID shows moderate payload variation, suggesting normal sensor fluctuations."
    else:
        interpretation = "The dominant ID shows high payload variance, suggesting significant variation in data or possible payload injection."
    
    return {
        "type": "payload_variance",
        "question": "What does the payload variance suggest about this window?",
        "answer": interpretation,
        "explanation": f"Dominant ID 0x{dom_id:03X} has payload variance: {var:.6f}"
    }


def generate_sa_irregular_id(df: pd.DataFrame, rng: np.random.Generator) -> Optional[dict]:
    """
    Which ID shows the most irregular timing pattern?
    """
    gaps_std = {}
    for id_val, group in df.groupby("ID"):
        ts = group["Timestamp"].to_numpy()
        if ts.size < 3:
            continue
        diffs = np.diff(ts)
        if diffs.size == 0:
            continue
        gaps_std[int(id_val)] = float(diffs.std())
    
    if not gaps_std:
        return None
    
    sorted_ids = sorted(gaps_std.items(), key=lambda x: x[1], reverse=True)
    correct_id = sorted_ids[0][0]
    std_val = sorted_ids[0][1]
    
    return {
        "type": "id_irregular_timing",
        "question": "Which ID shows the most irregular timing pattern?",
        "answer": f"0x{correct_id:03X}",
        "explanation": f"ID 0x{correct_id:03X} has the highest timing standard deviation: {std_val:.6f}s"
    }


def generate_sa_burst_explanation(stats: dict) -> Optional[dict]:
    """
    What best explains the burst behavior observed?
    """
    dom_share = stats.get("dominant_share", 0.0)
    gap_mean = stats.get("gap_mean", 0.0)
    gap_std = stats.get("gap_std", 0.0)
    
    if gap_mean < FLOODING_THRESHOLD and dom_share > 0.5:
        explanation = "Flooding from a compromised ECU - very small inter-frame gaps with high ID dominance"
    elif gap_mean > SUPPRESSION_THRESHOLD and gap_std > gap_mean * 0.5:
        explanation = "Diagnostic or recovery traffic causing bursts - large gaps with high variance"
    else:
        explanation = "Normal periodic behavior with minor bursts - typical timing patterns"
    
    return {
        "type": "burst_explanation",
        "question": "What best explains the burst behavior observed in this window?",
        "answer": explanation,
        "explanation": f"Dominant share: {dom_share:.2%}, Mean gap: {gap_mean:.6f}s, Gap std: {gap_std:.6f}s"
    }


def generate_sa_window_characterization(stats: dict) -> Optional[dict]:
    """
    How would you characterize this window overall?
    """
    dom_share = stats.get("dominant_share", 0.0)
    gap_std = stats.get("gap_std", 0.0)
    gap_mean = stats.get("gap_mean", 0.0)
    window_duration = stats.get("window_duration", 0.0)
    
    if dom_share > 0.7 and gap_std < FLOODING_THRESHOLD:
        characterization = "Highly irregular and unsafe - dominated by a single ID with very consistent flooding"
    elif dom_share < 0.4 and gap_std < gap_mean * 0.1:
        characterization = "Uniform and typical - well-distributed across IDs with consistent timing"
    else:
        characterization = "Mostly stable with minor anomalies - some irregularities but generally normal"
    
    return {
        "type": "overall_window",
        "question": "How would you characterize this window overall?",
        "answer": characterization,
        "explanation": f"Duration: {window_duration:.3f}s, Dominant share: {dom_share:.2%}, ID count: {len(stats.get('id_counts', []))}"
    }

# %%
# LLM Integration for Answer Enhancement
def enhance_answer_with_llm(question: str, rule_based_answer: str, 
                            explanation: str, context_preview: str) -> Optional[str]:
    """
    Use Ollama to generate a more comprehensive answer based on rule-based answer.
    Returns the LLM-generated answer or None if LLM is unavailable.
    """
    if not LLM_ENABLED:
        return None
    
    # Truncate context to avoid token limits
    context_preview = context_preview[:500] if context_preview else "N/A"
    
    prompt = f"""You are a CAN network security analyst. Given a question about CAN bus data analysis and an initial rule-based answer, 
provide a concise, comprehensive answer that builds upon the rule-based answer.

Question: {question}

Rule-based Answer: {rule_based_answer}

Data Context (first few lines):
{context_preview}

Provide a clear, detailed answer (max {MAX_ANSWER_TOKENS} words) that refines or expands on the rule-based answer. Be technical but accessible."""

    try:
        response = ollama.generate(
            model=LLM_MODEL,
            prompt=prompt,
            stream=False,
            options={
                "temperature": LLM_TEMPERATURE,
                "num_predict": MAX_ANSWER_TOKENS,
            }
        )
        
        llm_answer = response.get("response", "").strip()
        if llm_answer:
            # Clean up the answer
            llm_answer = llm_answer.replace("\n", " ")
            # Limit to max tokens as a safety measure
            words = llm_answer.split()[:MAX_ANSWER_TOKENS]
            llm_answer = " ".join(words)
            return llm_answer
    except Exception as e:
        print(f"[WARN] LLM call failed: {e}")
        return None
    
    return None


def enhance_sa_with_llm(sa: dict, context_preview: str) -> dict:
    """
    Enhance a short answer question with LLM-generated answer.
    Stores both rule-based and LLM answers in the question object.
    """
    if not LLM_ENABLED:
        return sa
    
    llm_answer = enhance_answer_with_llm(
        question=sa["question"],
        rule_based_answer=sa["answer"],
        explanation=sa.get("explanation", ""),
        context_preview=context_preview
    )
    
    if llm_answer:
        sa["llm_answer"] = llm_answer
    
    return sa


# %%
# ==== Optional: enrich profiles with global baseline ID rates ====
# Run once after datasets, profiles are created.
for name, df_full in datasets.items():
    id_counts_full = df_full["ID"].value_counts()
    total_full = len(df_full)
    baseline_rates = {int(i): float(c / total_full) for i, c in id_counts_full.items()}
    profiles[name]["baseline_id_rate"] = baseline_rates


def generate_sa_for_window(
    df: pd.DataFrame,
    profile: dict,
    baseline: BaselineStats,
    rng: np.random.Generator,
    metrics: Optional[dict] = None,
) -> List[dict]:
    """Generate short answer questions for a single window (SA_Quesiton.csv aligned)."""
    metrics = metrics or compute_window_metrics(df, baseline)
    if not metrics:
        return []

    format_values = {
        "ATTACK_BURST_MIN_LEN": ATTACK_BURST_MIN_LEN,
        "ATTACK_SUSTAIN_LEN": ATTACK_SUSTAIN_LEN,
        "MAX_ID_SHARE_LOW": ratio_to_str(MAX_ID_SHARE_LOW),
        "MAX_ID_SHARE": ratio_to_str(MAX_ID_SHARE),
        "BASELINE_P95": f"{baseline.frame_rate_p95:.4f}",
        "BASELINE_P99": f"{baseline.frame_rate_p99:.4f}",
        "FPS_SUBWINDOW": FPS_SUBWINDOW,
        "FLOODING_THRESHOLD": f"{FLOODING_THRESHOLD:g}",
        "FLOODING_STD": f"{FLOODING_STD:g}",
        "SUPPRESSION_THRESHOLD": f"{SUPPRESSION_THRESHOLD:g}",
        "JITTER_RATIO": f"{JITTER_RATIO:g}",
        "PAYLOAD_VAR_LOW": f"{baseline.payload_var_low:.6f}",
        "PAYLOAD_VAR_HIGH": f"{baseline.payload_var_high:.6f}",
        "HIGH_DLC_LOW": ratio_to_str(baseline.high_dlc_low),
        "HIGH_DLC_MID": ratio_to_str(baseline.high_dlc_mid),
        "HIGH_DLC_HIGH": ratio_to_str(baseline.high_dlc_high),
        "CRITICAL_RATE_P95": f"{max(baseline.critical_rate_p95_by_id.values() or [0.0]):.6f}",
        "CRITICAL_SPIKE_MAX_COUNT": CRITICAL_SPIKE_MAX_COUNT,
        "P1": BASELINE_P1_PERCENTILE,
        "P99": BASELINE_P99_PERCENTILE,
        "OUT_OF_RANGE_ID_RATIO_HIGH": ratio_to_str(baseline.out_of_range_id_ratio_high),
        "TIMESTAMP_BUCKET_SHARE_THRESHOLD": ratio_to_str(TIMESTAMP_BUCKET_SHARE_THRESHOLD),
        "INDICATOR_SET": "attack_present, dominant_id, high_dlc, out_of_range, timing, timestamp",
    }

    sas: List[dict] = []

    def add_sa(sa_type: str, question: str, answer: str, explanation: str) -> None:
        sas.append({
            "type": sa_type,
            "question": question,
            "answer": answer,
            "explanation": explanation.format_map(format_values),
        })

    attack_ratio = ratio_to_percent_str(metrics["attack_ratio"])
    add_sa(
        "attack_ratio",
        "What percentage of frames are attack-labeled in this window?",
        f"{attack_ratio}%",
        "attack_ratio = (count of non-zero Flag frames / total frames) * 100.",
    )

    if metrics["attack_count"] == 0:
        attack_pattern = "Clean"
    elif metrics["attack_max_run"] < ATTACK_BURST_MIN_LEN:
        attack_pattern = "Sporadic"
    elif metrics["attack_max_run"] < ATTACK_SUSTAIN_LEN:
        attack_pattern = "Bursty"
    else:
        attack_pattern = "Sustained"
    add_sa(
        "attack_pattern",
        "What attack pattern does this window show?",
        attack_pattern,
        "attack_pattern: Clean if attack_count=0; Sporadic if attack_count>0 and max_run<{ATTACK_BURST_MIN_LEN}; Bursty if {ATTACK_BURST_MIN_LEN}≤max_run<{ATTACK_SUSTAIN_LEN}; Sustained if max_run≥{ATTACK_SUSTAIN_LEN}. attack_count = count(Flag!=0); max_run = longest consecutive non-zero Flag run.",
    )

    dominant_id = metrics["dominant_id"]
    add_sa(
        "dominant_id",
        "Which CAN ID appears most frequently in this window?",
        f"0x{dominant_id:03X}",
        "Select the ID with the highest frame count (tie-break: smallest ID).",
    )

    max_share = metrics["max_share"]
    if max_share <= MAX_ID_SHARE_LOW:
        id_concentration = "Balanced"
    elif max_share <= MAX_ID_SHARE:
        id_concentration = "Skewed"
    else:
        id_concentration = "Dominant"
    add_sa(
        "id_concentration",
        "How concentrated is traffic across CAN IDs in this window?",
        id_concentration,
        "id_concentration: Balanced if max_share≤{MAX_ID_SHARE_LOW}; Skewed if {MAX_ID_SHARE_LOW}<max_share≤{MAX_ID_SHARE}; Dominant if max_share>{MAX_ID_SHARE}.",
    )

    add_sa(
        "fps",
        "What is the frame rate in this window (fps)?",
        f"{metrics['fps']:.4f}",
        "fps = total frames / window duration.",
    )

    fps = metrics["fps"]
    traffic_shift = "No Change"
    if fps <= baseline.frame_rate_p95:
        traffic_shift = "No Change"
    elif (baseline.frame_rate_p95 < fps <= baseline.frame_rate_p99) and (metrics["fps_second_half"] > metrics["fps_first_half"]):
        traffic_shift = "Gradual Increase"
    elif (metrics["max_subwindow_fps"] > baseline.frame_rate_p99) and (metrics["fps_first_half"] <= baseline.frame_rate_p95):
        traffic_shift = "Sudden Spike"
    elif (fps > baseline.frame_rate_p99) and (metrics["fps_second_half"] > baseline.frame_rate_p99):
        traffic_shift = "Sustained High"
    add_sa(
        "traffic_shift",
        "What change in traffic intensity is observed in this window relative to baseline?",
        traffic_shift,
        "traffic_shift: No Change if fps≤{BASELINE_P95}; Gradual Increase if {BASELINE_P95}<fps≤{BASELINE_P99} and fps_second_half>fps_first_half; Sudden Spike if max_subwindow_fps>{BASELINE_P99} and fps_first_half≤{BASELINE_P95}; Sustained High if fps>{BASELINE_P99} and fps_second_half>{BASELINE_P99}. fps_first_half/second_half computed on the first/second half of the window; max_subwindow_fps computed over sliding windows of {FPS_SUBWINDOW} frames.",
    )

    add_sa(
        "max_gap",
        "What is the maximum inter-frame gap in this window (seconds)?",
        f"{metrics['max_gap']:.6f}",
        "max_gap = max(timestamp[i] - timestamp[i-1]) over the window.",
    )

    if metrics["max_gap"] > SUPPRESSION_THRESHOLD:
        timing_cause = "Suppression"
    elif metrics["gap_mean"] < FLOODING_THRESHOLD and metrics["gap_std"] < FLOODING_STD:
        timing_cause = "Flooding"
    elif metrics["gap_mean"] > 0 and metrics["gap_std"] > JITTER_RATIO * metrics["gap_mean"]:
        timing_cause = "Jitter"
    else:
        timing_cause = "None"
    add_sa(
        "timing_cause",
        "Which timing irregularity is most responsible for the observed behavior in this window?",
        timing_cause,
        "timing_cause: Suppression if max_gap>{SUPPRESSION_THRESHOLD}; Flooding if gap_mean<{FLOODING_THRESHOLD} and gap_std<{FLOODING_STD}; Jitter if gap_std>{JITTER_RATIO}*gap_mean; None otherwise. gap_mean/gap_std over inter-frame gaps; {FLOODING_STD} and {JITTER_RATIO} are fixed thresholds.",
    )

    add_sa(
        "dominant_var",
        "What is the dominant CAN ID's payload variance in this window?",
        f"{metrics['dominant_var']:.6f}",
        "dominant_var = variance of payload bytes for the most frequent ID.",
    )

    if metrics["dominant_var"] <= baseline.payload_var_low:
        payload_behavior = "Static Signal"
    elif metrics["dominant_var"] <= baseline.payload_var_high:
        payload_behavior = "Controlled Variation"
    else:
        payload_behavior = "Erratic Change"
    add_sa(
        "payload_behavior",
        "What does the observed payload variation suggest about signal behavior in this window?",
        payload_behavior,
        "payload_behavior: Static Signal if dominant_var≤{PAYLOAD_VAR_LOW}; Controlled Variation if {PAYLOAD_VAR_LOW}<var≤{PAYLOAD_VAR_HIGH}; Erratic Change if var>{PAYLOAD_VAR_HIGH}.",
    )

    add_sa(
        "dlc_mode",
        "What is the most common DLC value in this window?",
        f"{metrics['dlc_mode']}",
        "dlc_mode = mode of DLC values in the window.",
    )

    if metrics["high_dlc_share"] <= baseline.high_dlc_low:
        dlc_behavior = "Typical"
    elif metrics["high_dlc_share"] <= baseline.high_dlc_mid:
        dlc_behavior = "Occasional High DLC"
    elif metrics["high_dlc_share"] <= baseline.high_dlc_high:
        dlc_behavior = "Concentrated High DLC"
    else:
        dlc_behavior = "Protocol Atypical"
    add_sa(
        "dlc_behavior",
        "How does DLC usage in this window differ from typical protocol behavior?",
        dlc_behavior,
        "dlc_behavior: Typical if high_DLC_share≤{HIGH_DLC_LOW}; Occasional High DLC if {HIGH_DLC_LOW}<share≤{HIGH_DLC_MID}; Concentrated High DLC if {HIGH_DLC_MID}<share≤{HIGH_DLC_HIGH}; Protocol Atypical if share>{HIGH_DLC_HIGH}.",
    )

    add_sa(
        "missing_count",
        "How many expected control IDs are missing from this window?",
        f"{metrics['missing_count']}",
        "missing_count = count of expected baseline IDs not present in the window.",
    )

    critical_above = metrics["critical_above"]
    if not critical_above:
        critical_behavior = "Normal Operation"
    elif len(critical_above) == 1:
        cid = critical_above[0]
        count = metrics["critical_counts"].get(cid, 0)
        if count <= CRITICAL_SPIKE_MAX_COUNT:
            critical_behavior = "Localized Spike"
        else:
            critical_behavior = "Sustained Overuse"
    else:
        critical_behavior = "Coordinated Activation"
    add_sa(
        "critical_behavior",
        "Which safety-critical message behavior best explains this window?",
        critical_behavior,
        "critical_behavior: Normal Operation if max_critical_rate≤{CRITICAL_RATE_P95}; Localized Spike if exactly one critical ID rate>{CRITICAL_RATE_P95} and its count≤{CRITICAL_SPIKE_MAX_COUNT}; Sustained Overuse if exactly one critical ID rate>{CRITICAL_RATE_P95} and its count>{CRITICAL_SPIKE_MAX_COUNT}; Coordinated Activation if ≥2 critical IDs rates>{CRITICAL_RATE_P95}.",
    )

    add_sa(
        "out_of_range_id_count",
        "How many CAN IDs show payload values outside their baseline range?",
        f"{metrics['out_of_range_id_count']}",
        "out_of_range_id_count = number of IDs with any payload byte outside baseline {P1}–{P99} bounds.",
    )

    ratio = metrics["out_of_range_id_ratio"]
    if ratio == 0:
        plausibility_relation = "Baseline Conformant"
    elif ratio <= baseline.out_of_range_id_ratio_high:
        plausibility_relation = "Isolated Deviation"
    else:
        plausibility_relation = "Systematic Deviation"
    add_sa(
        "plausibility_relation",
        "How do observed payload deviations relate to baseline signal expectations?",
        plausibility_relation,
        "plausibility_relation: Baseline Conformant if out_of_range_id_ratio=0; Isolated Deviation if 0<ratio≤{OUT_OF_RANGE_ID_RATIO_HIGH}; Systematic Deviation if ratio>{OUT_OF_RANGE_ID_RATIO_HIGH}.",
    )

    add_sa(
        "duplicate_count",
        "How many exact duplicate frames occur in this window?",
        f"{metrics['duplicate_count']}",
        "duplicate_count = count of duplicates with identical Timestamp, ID, DLC, payload, and Flag.",
    )

    if metrics["backward_steps"] and metrics["bucket_share"] >= TIMESTAMP_BUCKET_SHARE_THRESHOLD:
        timestamp_cause = "Mixed Issue"
    elif metrics["backward_steps"]:
        timestamp_cause = "Backward Jump"
    elif metrics["bucket_share"] >= TIMESTAMP_BUCKET_SHARE_THRESHOLD:
        timestamp_cause = "Clock Bucketing"
    else:
        timestamp_cause = "Normal"
    add_sa(
        "timestamp_cause",
        "What logging or synchronization issue best explains the observed timestamp pattern?",
        timestamp_cause,
        "timestamp_cause: Normal if no backward steps and bucket_share<{TIMESTAMP_BUCKET_SHARE_THRESHOLD}; Backward Jump if any backward step and bucket_share<{TIMESTAMP_BUCKET_SHARE_THRESHOLD}; Clock Bucketing if no backward step and bucket_share≥{TIMESTAMP_BUCKET_SHARE_THRESHOLD}; Mixed Issue if both occur.",
    )

    add_sa(
        "attack_type",
        "What attack type best fits the behavior observed in this window?",
        profile.get("attack_label", "DoS"),
        "attack_type = dataset label (DoS/Fuzzy/Gear/RPM).",
    )

    indicator_count = metrics["indicator_count"]
    if indicator_count == 0:
        evidence_pattern = "No Indicator"
    elif indicator_count == 1:
        evidence_pattern = "Single Indicator"
    elif metrics["contradictions"]:
        evidence_pattern = "Conflicting Evidence"
    elif metrics["indicator_count_second_half"] > metrics["indicator_count_first_half"]:
        evidence_pattern = "Temporal Escalation"
    else:
        evidence_pattern = "Multi-Signal Consistency"
    add_sa(
        "evidence_pattern",
        "Which combination of indicators most strongly supports the attack interpretation for this window?",
        evidence_pattern,
        "evidence_pattern: No Indicator if indicator_count=0; Single Indicator if indicator_count=1; Conflicting Evidence if both attack-leaning and normal-leaning indicators present; Temporal Escalation if indicator_count increases from first half to second half; Multi-Signal Consistency if indicator_count≥2 and no contradictions. indicator_count is computed over {INDICATOR_SET} with per-indicator thresholds defined in the set.",
    )

    if len(sas) > SHORT_ANSWERS_PER_WINDOW:
        idxs = rng.choice(len(sas), size=SHORT_ANSWERS_PER_WINDOW, replace=False)
        sas = [sas[i] for i in idxs]

    return sas


In [6]:
# Cell 5: main loop – per-dataset short answer question generation with LLM enhancement
global_question_counts: Dict[str, int] = {}
global_question_types: Dict[str, str] = {}
global_total = 0

for ds_idx, (name, df) in enumerate(datasets.items()):
    print(f"[INFO] Generating short answer questions for {name}")
    starts = iter_window_starts(len(df))
    sampled_starts_cache_path = CACHE_DIR / f"sampled_starts_{name}_{run_hash}.pkl"
    cached_sampled_starts = load_cache(sampled_starts_cache_path, None)
    if cached_sampled_starts is None:
        sampled_starts = sample_window_indices(starts, rng_global)
        save_cache(sampled_starts_cache_path, sampled_starts)
    else:
        _ = sample_window_indices(starts, rng_global)
        sampled_starts = cached_sampled_starts
        print(f"[INFO] Loaded sampled starts -> {sampled_starts_cache_path}")

    window_stats_cache_path = CACHE_DIR / f"window_stats_{name}_{run_hash}.pkl"
    context_cache_path = CACHE_DIR / f"context_{name}_{run_hash}.pkl"
    window_stats_cache = load_cache(window_stats_cache_path, {})
    context_cache = load_cache(context_cache_path, {})

    out_dir = Path(f"{name}_sa_qa")
    questions_dir = out_dir / "questions"
    llm_dir = out_dir / "llm_answers"
    questions_dir.mkdir(parents=True, exist_ok=True)
    llm_dir.mkdir(parents=True, exist_ok=True)
    ql_path = questions_dir / f"{name.lower()}_sa_questions.jsonl"
    qj_path = questions_dir / f"{name.lower()}_sa_questions.json"
    llm_path = llm_dir / f"{name.lower()}_sa_llm_answers.jsonl"

    ql_path.write_text("", encoding="utf-8")
    llm_path.write_text("", encoding="utf-8")

    qa_id_counter = 0
    records_all = []
    llm_records = []
    question_counts: Dict[str, int] = {}
    question_types: Dict[str, str] = {}

    with ql_path.open("a", encoding="utf-8") as f:
        for window_idx, start in enumerate(tqdm(sampled_starts, desc=f"{name} windows")):
            window = df.iloc[start:start + WINDOW_SIZE].copy().reset_index(drop=True)
            metrics = window_stats_cache.get(start)
            if metrics is None:
                metrics = compute_window_metrics(window, baseline)
                window_stats_cache[start] = metrics
            sa_items = generate_sa_for_window(window, profiles[name], baseline, rng_global, metrics=metrics)
            if not sa_items:
                continue

            context = context_cache.get(start)
            if context is None:
                context = format_window(window)
                context_cache[start] = context
            context_preview = "\n".join(context.split("\n")[:5])  # First 5 lines for LLM
            
            for local_q_idx, sa in enumerate(sa_items):
                # Enhance answer with LLM if enabled
                if LLM_ENABLED:
                    sa = enhance_sa_with_llm(sa, context_preview)
                
                qa_id = f"{name}_SA_{window_idx:06d}_{local_q_idx:02d}"
                record = {
                    "qa_id": qa_id,
                    "metadata": {
                        "dataset": name,
                        "window_index": int(window_idx),
                        "window_start": int(start),
                        "window_size": int(WINDOW_SIZE),
                    },
                    "context": context,
                    "sa_type": sa["type"],
                    "question": sa["question"],
                    "ground_truth": sa["answer"],
                    "explanation": sa.get("explanation", ""),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                records_all.append(record)
                qa_id_counter += 1
                question_counts[sa["question"]] = question_counts.get(sa["question"], 0) + 1
                question_types[sa["question"]] = sa["type"]
                global_question_counts[sa["question"]] = global_question_counts.get(sa["question"], 0) + 1
                global_question_types[sa["question"]] = sa["type"]
                global_total += 1

                if LLM_ENABLED and sa.get("llm_answer"):
                    llm_record = {
                        "qa_id": qa_id,
                        "question": sa["question"],
                        "llm_answer": sa["llm_answer"],
                    }
                    llm_records.append(llm_record)

    with qj_path.open("w", encoding="utf-8") as jf:
        json.dump(records_all, jf, ensure_ascii=False, indent=2)

    if llm_records:
        with llm_path.open("a", encoding="utf-8") as lf:
            for rec in llm_records:
                lf.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {name}: saved {qa_id_counter} short answer questions -> {ql_path}, {qj_path}")
    save_cache(window_stats_cache_path, window_stats_cache)
    save_cache(context_cache_path, context_cache)

if global_total > 0:
    detail_path = Path("sa_question_summary.csv")
    with detail_path.open("w", encoding="utf-8", newline="") as f_detail:
        writer = csv.writer(f_detail)
        writer.writerow(["SA_Type", "Question", "Count", "Ratio"])
        for question, count in sorted(global_question_counts.items()):
            sa_type = global_question_types.get(question, "")
            ratio = count / global_total if global_total else 0
            writer.writerow([sa_type, question, count, f"{ratio:.6f}"])
    print(f"[INFO] Summary -> {detail_path}")

[INFO] Generating short answer questions for DoS


DoS windows: 100%|██████████| 916/916 [01:27<00:00, 10.50it/s]


[INFO] DoS: saved 916 short answer questions -> DoS_sa_qa\questions\dos_sa_questions.jsonl, DoS_sa_qa\questions\dos_sa_questions.json
[INFO] Generating short answer questions for Fuzzy


Fuzzy windows: 100%|██████████| 959/959 [02:10<00:00,  7.35it/s]


[INFO] Fuzzy: saved 959 short answer questions -> Fuzzy_sa_qa\questions\fuzzy_sa_questions.jsonl, Fuzzy_sa_qa\questions\fuzzy_sa_questions.json
[INFO] Generating short answer questions for Gear


Gear windows: 100%|██████████| 1110/1110 [01:38<00:00, 11.22it/s]


[INFO] Gear: saved 1110 short answer questions -> Gear_sa_qa\questions\gear_sa_questions.jsonl, Gear_sa_qa\questions\gear_sa_questions.json
[INFO] Generating short answer questions for RPM


RPM windows: 100%|██████████| 1155/1155 [01:44<00:00, 11.09it/s]


[INFO] RPM: saved 1155 short answer questions -> RPM_sa_qa\questions\rpm_sa_questions.jsonl, RPM_sa_qa\questions\rpm_sa_questions.json
[INFO] Summary -> sa_question_summary.csv
